<a href="https://colab.research.google.com/github/BandaAkshitha/Natural-Language-Processing/blob/main/NLP_Lab15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TASK-1

Import Libraries

In [35]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np

Load ELMO Model

In [36]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Text Corpus

In [37]:
sentences = ["The bank will not approve the loan",
             "He sat on the river bank"]

Generate Embeddings

In [38]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(2, 7, 1024)


Inspect Word Embedding (“bank”)

In [39]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank"
idx1 = tokenized[0].index("bank")
idx2 = tokenized[1].index("bank")

# Extract embeddings
bank_emb_1 = embeddings[0][idx1]
bank_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.9086765  -0.36578754 -0.07339765  0.586675    0.22132948 -0.7657321
 -0.11816446 -0.30227882 -0.06137528  0.1695182 ], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.15614763  0.28345656 -0.08914638  0.2069506   0.1524184  -0.2037305
 -0.10812807  0.03747292  0.5214868   0.35352594], shape=(10,), dtype=float32)


Compare Context (Cosine Similarity)

In [40]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bank' meanings:", sim)

Similarity between 'bank' meanings: 0.57813895


TASK-2

BERT model Implementation

In [41]:
from transformers import BertTokenizer, BertModel
import torch
from torch.nn.functional import cosine_similarity

# Load model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

sentences = [
    "The bank will not approve the loan",
    "He sat on the river bank"
]

embeddings = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors='pt')
    outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    for i, token in enumerate(tokens):
        if token == 'bank':
            embeddings.append(token_embeddings[0][i])
            break

# Cosine similarity
sim = cosine_similarity(embeddings[0].unsqueeze(0),
                        embeddings[1].unsqueeze(0))

print("Cosine Similarity:", sim.item())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine Similarity: 0.3052193820476532


TASK-3

BERT Model Implementation

In [42]:
sentences = [
    "The bat is flying",
    "He hit the ball with a bat"
]

In [43]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

Preprocess Input

In [44]:
inputs = preprocess(sentences)
print(inputs.keys())

dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


Generate Embeddings

In [45]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


Get Tokens (Important Step)

In [46]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[ 101 1996 7151 2003 3909  102    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 101 2002 2718 1996 3608 2007 1037 7151  102    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    

Extract bat Embeddings

In [47]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-6.0895458e-05  2.6044571e-01  1.2677926e-01 -2.8091672e-01
  1.3292238e-02 -4.8430103e-01  7.0899075e-01  7.3061740e-01
  2.0883048e-01 -7.8647310e-01], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.06056872 -0.90724915 -0.06744897 -0.27453673 -0.4663377   0.04165859
  0.23526248  0.35729024  0.9171281  -0.67234385], shape=(10,), dtype=float32)


Cosine Similarity

In [48]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.4342774


TASK-4

ELMO model Implementation

In [49]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load ELMo from TensorFlow Hub
elmo = hub.load("https://tfhub.dev/google/elmo/3")

sentences = [
    ["The", "bat", "is", "flying"],
    ["He", "hit", "the", "ball", "with", "a", "bat"]
]

embeddings = elmo.signatures["default"](
    tf.constant([" ".join(s) for s in sentences]) # Pass the joined sentences as a tf.constant directly
)["elmo"]

bat_vectors = []

for i, sentence in enumerate(sentences):
    for j, word in enumerate(sentence):
        if word == "bat":
            bat_vectors.append(embeddings[i][j].numpy())
            break

# Cosine similarity
sim = cosine_similarity([bat_vectors[0]], [bat_vectors[1]])

print("Cosine Similarity:", sim[0][0])

Cosine Similarity: 0.648854


TASK-5

Text Classification Using ELMO+Naive Bayes

Import Libraries

In [50]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Text Corpus

In [51]:
sentences = [
    "Win money now",
    "Claim your prize",
    "Hello how are you",
    "Let's meet tomorrow",
    "Free lottery ticket",
    "Are you coming today"
]

labels = [1, 1, 0, 0, 1, 0]   # 1 = spam, 0 = normal

Load ELMO Model

In [52]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Generate ELMo Embeddings

In [53]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (6, 4, 1024)


Convert to Sentence Embeddings

In [54]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 1024)


Train-Test Split

In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model

In [56]:
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

Model Testing

In [57]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [0 0]
Actual: [1 1]


Model Evaluation

In [58]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.0


Prediction on New Text

In [59]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

Prediction: Not Spam


TASK-6

Text Classification using BERT+NB

In [60]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [61]:
bert_inputs = preprocess(sentences)

In [62]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)

BERT Accuracy: 0.0
